# __Homework 4:__ Practical analysis with BioPython

For the homework, you are going to extend the code from the analysis of our FASTQ file in lectures 8 and 9.
Recall that the FASTQ file contains reads from a real sequencing run of influenza virus HA and NA genes.

---
The __actual sequences__ are as follows:

    5'-[end of HA]-AGGCGGCCGC-[16 X N barcode]-3'
or 

    5'-[end of NA]-AGGCGGCCGC-[16 X N barcode]-3'
---


__The end of NA is__ `...CACGATAGATAAATAATAGTGCACCAT`
    
__The end of HA is__ `...CCGGATTTGCATATAATGATGCACCAT`

---    

    
The __sequencing reads__ from the reverse end of the molecules (in 5'>3' orientation), so the sequencing reads are as follows:

    5'-[reverse complement of 16 X N barcode]-GCGGCCGCCT-[reverse complement of the end of HA]-3'
or

    5'-[reverse complement of 16 X N barcode]-GCGGCCGCCT-[reverse complement of the end of NA]-3'

---   
    
The reads can originate from **either** HA or NA, and that will be distinguished by the most 3' end of the read.
But in our example exercise in class, we did not distinguish among reads matching to HA and NA, as we didn't even look far enough into the read to tell the identity.

For the homework, your goal is to write code that extends the material from lectures 8 and 9 to also distinguish between HA and NA.
This homework can be completed almost entirely by re-using code from lecture 9. You will need to set up your analysis to do the following:
 1. Get the reverse complement of each read.
 2. Determine if it matches the expected pattern for HA and NA, and if so which one.
 3. If it matches, extract the barcode and add it to a dictionary to keep track of counts.
 4. Determine the number and distribution of barcodes for HA and NA separately.

Please include code to address each of the following questions. Please include code comments to explain what your code is attempting to accomplish. Don't forget to include references to the sources you used to obtain your answer, including your classmates (if you are working in groups).  

1. How many reads map to HA, and how many reads map to NA?

   Please __include at least two test cases__ showing that your HA/NA classifier works. You can copy the `run_tests` helper from lecture 9, or use `assert` statements - either is fine. These test cases are part of the points for this question.


In [ ]:
import re
import Bio.SeqIO
import Bio.Seq

# reading fastq files
seqreads = list(Bio.SeqIO.parse("barcodes_R1.fastq", format = "fastq"))

#adding gene-specific sequences
ha_end = "CCGGATTTGCATATAATGATGCACCAT"
na_end = "CACGATAGATAAATAATAGTGCACCAT"
upstream = "AGGCGGCCGC"

#finding the reverse complement of each read and then seeing if it belongs to HA or NA
#returns none of both or none are found
def classify_gene(seqread):
    seq = str(Bio.Seq.Seq(str(seqread)).reverse_complement()).upper()
    is_ha = ha_end in seq
    is_na = na_end in seq
    if is_ha and not is_na:
        return "HA"
    if is_na and not is_ha:
        return "NA"
    return None

#using artificial sequence to see if the function can correctly identify HA/NA/neither
# Used help from ChatGPT
ha_test = Bio.Seq.Seq(ha_end + upstream + "AAAAAAAAAAAAAAAA").reverse_complement()
na_test = Bio.Seq.Seq(na_end + upstream + "CCCCCCCCCCCCCCCC").reverse_complement()
assert classify_gene(ha_test) == "HA"
assert classify_gene(na_test) == "NA"
assert classify_gene("ACGTACGTACGT") is None
print("Classifier works")

# reusing read_barcode function from lecture 9
def read_barcode(seqread, bclen=16, upstream="AGGCGGCCGC"):
    seqread_upper = str(seqread).upper()
    seqread_original = str(Bio.Seq.Seq(seqread_upper).reverse_complement())

    barcode_pattern = re.compile(f"{upstream}(?P<barcode>[ACTGN]{{{bclen}}})$")

    match = barcode_pattern.search(seqread_original)

    if match:
        return match.group("barcode")
    else:
        return None

# counting reads by gene and counting each barcode separately
read_counts = {"HA": 0, "NA": 0}
invalid_counts = {"HA": 0, "NA": 0}
barcode_counts = {"HA": {}, "NA": {}}
unclassified = 0

for read in seqreads:
    gene = classify_gene(read.seq)
    if gene is None:
        unclassified += 1
        continue
    read_counts[gene] += 1
    barcode = read_barcode(read.seq)
    if barcode is None: #used help from ChatGPT for this logic
        invalid_counts[gene] += 1
    else:
        if barcode not in barcode_counts[gene]:
            barcode_counts[gene][barcode] = 0
        barcode_counts[gene][barcode] += 1

print("HA reads:", read_counts["HA"])
print("NA reads:", read_counts["NA"])
print("Unclassified reads:", unclassified)


Classifier works
HA reads: 5409
NA reads: 4122
Unclassified reads: 469


2. How many HA sequences did not have a valid barcode? Also anwer the same question for NA.

In [15]:
# your code here...
# using for loop at end of last question:
print("HA reads without valid barcodes:", invalid_counts["HA"])
print("NA reads without valid barcodes:", invalid_counts["NA"])

HA reads without valid barcodes: 163
NA reads without valid barcodes: 212


3. What is the HA barcode with the most counts (and how many counts)? Also answer the same question for NA.

    _Hint: you will need to find the key associated with the maximum value in your dictionary. There are many ways to do this._

In [16]:
# your code here...
# establish ha/na_top as top barcode using max(dictionary, key=dictionary.get (figured out these keys using ChatGPT)
ha_top = max(barcode_counts["HA"], key=barcode_counts["HA"].get)
na_top = max(barcode_counts["NA"], key=barcode_counts["NA"].get)
print("Most common HA barcode:", ha_top, "-", barcode_counts["HA"][ha_top], "reads")
print("Most common NA barcode:", na_top, "-", barcode_counts["NA"][na_top], "reads")

Most common HA barcode: CCCGACCCGACATTAA - 155 reads
Most common NA barcode: ACCAGTTCTCCCCGGG - 152 reads
